In [ ]:
import os

# 1. Force WSLg to use X11 (XWayland) to fix the GLFW window position bug
os.environ['DISPLAY'] = ':0'
if 'WAYLAND_DISPLAY' in os.environ:
    del os.environ['WAYLAND_DISPLAY']
os.environ['MUJOCO_GL'] = 'glfw'

import mujoco
import mujoco.viewer
import numpy as np
import time

original_cwd = os.getcwd()
panda_dir = "mujoco_menagerie/franka_emika_panda"
os.chdir(panda_dir)

xml_string = """
<mujoco>
  <include file="scene.xml"/>
  <worldbody>
    <body name="ball" pos="0.5 0 0.5">
      <freejoint name="ball_joint"/>
      <geom type="sphere" size="0.05" rgba="1 0 0 1"/>
    </body>
  </worldbody>
</mujoco>
"""

temp_xml_path = "temp_scene.xml"
with open(temp_xml_path, "w") as f:
    f.write(xml_string)

model = mujoco.MjModel.from_xml_path(temp_xml_path)
data = mujoco.MjData(model)

os.remove(temp_xml_path)
os.chdir(original_cwd)

steps = 500
t = np.linspace(0, np.pi, steps)
x_traj = 0.5 + 0.2 * np.cos(t)
y_traj = 0.2 * np.sin(t)
z_traj = 0.5 + 0.2 * np.sin(t)

ball_qpos_adr = model.jnt_qposadr[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, "ball_joint")]

with mujoco.viewer.launch_passive(model, data) as viewer:
    for i in range(steps):
        data.qpos[ball_qpos_adr]     = x_traj[i]
        data.qpos[ball_qpos_adr + 1] = y_traj[i]
        data.qpos[ball_qpos_adr + 2] = z_traj[i]
        
        mujoco.mj_step(model, data)
        viewer.sync()
        time.sleep(model.opt.timestep)
        
    # 2. KEEP ALIVE LOOP: Prevents the window from freezing after the rollout
    print("Trajectory complete! You can now pan/rotate the camera.")
    while viewer.is_running():
        time.sleep(0.1)